# The Overshoot Deficit — reproduction notebook

Son Do · Noralabs, Vietnam

Everything in the manuscript that can be computed is computed here, from one data file, with no
hard-coded results. Run top to bottom; the last cell checks every number in the paper against what
this notebook just produced and fails loudly on any mismatch.

**Why the notebook exists.** An audit on 30 August 2026 found that three of the four figures had been
generated from an earlier data panel of 1,255 sessions while every table used the current panel of
1,340, and that the figure values were pasted constants rather than computed ones. Both faults are
structural, not clerical: they are what happens when figures and tables can disagree about which file
they read. One `PANEL` constant and one consistency cell remove that possibility.

**Not included.** The fitted classifier (the `A+M` branch) depends on frozen model artefacts and is
loaded from its stored output where the paper needs it; those rows are labelled as such rather than
silently recomputed. The directional-change rule, which carries the paper's headline claim, is
reproduced here in full.


## 0 Configuration — the single source of truth


In [2]:
import numpy as np, pandas as pd, json, hashlib, warnings
from scipy import stats
from scipy.optimize import curve_fit
from numba import njit
warnings.filterwarnings('ignore')

PANEL     = 'panel2.npz'        # the ONLY data file this notebook reads
THETAS    = (5e-4, 1e-3, 2e-3, 5e-3, 7e-3, 1e-2)      # frozen, manuscript Section 2
THETA_FIG = (2e-4,) + THETAS                          # Figure 1 adds one narrower point
EDGES     = (0.05, 0.10, 0.15, 0.22, 0.32, 0.45, 0.65)
MIN_LEGS  = 60
COST      = 0.175               # index points per unit of position change
DELAY     = 1                   # seconds between confirmation and fill
SESSION   = (91500, 144500)     # sample window, HHMMSS
BREAK     = (113000, 130000)    # midday break, no observations in between

h = hashlib.sha256(open(PANEL,'rb').read()).hexdigest()
print(f'panel   : {PANEL}')
print(f'sha256  : {h}')


panel   : panel2.npz
sha256  : 969ed5b907388549d857f08cc705e6b3fdebe50f2332aacf36c102f75765a900


## 1 Directional-change engine

Compiled event decomposition. For each confirmed trend it records the previous extreme, the extreme,
the confirmation index and the direction, using only information available at the confirmation tick.


In [4]:
@njit  # no cache: numba cannot locate a notebook cell on disk
def _ev(prices, theta, out_i, out_v, cap):
    n = prices.shape[0]; k = 0
    ext_p = prices[0]; ext_i = 0
    curr_max = prices[0]; curr_min = prices[0]
    tp_max = 0; tp_min = 0; up = True
    for i in range(n):
        if up:
            if prices[i] < (1.0 - theta) * curr_max:
                if k < cap:
                    out_i[k,0]=ext_i; out_i[k,1]=tp_max; out_i[k,2]=i; out_i[k,3]=1
                    out_v[k,0]=(curr_max-ext_p)/(ext_p*theta); out_v[k,1]=tp_max-ext_i; k+=1
                ext_p=curr_max; ext_i=tp_max; up=False; curr_min=prices[i]; tp_min=i
            elif prices[i] > curr_max:
                curr_max=prices[i]; tp_max=i
        else:
            if prices[i] > (1.0 + theta) * curr_min:
                if k < cap:
                    out_i[k,0]=ext_i; out_i[k,1]=tp_min; out_i[k,2]=i; out_i[k,3]=-1
                    out_v[k,0]=(curr_min-ext_p)/(ext_p*theta); out_v[k,1]=tp_min-ext_i; k+=1
                ext_p=curr_min; ext_i=tp_min; up=True; curr_max=prices[i]; tp_max=i
            elif prices[i] < curr_min:
                curr_min=prices[i]; tp_min=i
    return k

def events(prices, theta, offset=0):
    p=np.ascontiguousarray(np.asarray(prices,np.float64)); n=p.shape[0]; cap=max(16,n)
    oi=np.zeros((cap,4),np.int64); ov=np.zeros((cap,2),np.float64)
    k=_ev(p,float(theta),oi,ov,cap); oi=oi[:k].copy(); ov=ov[:k].copy(); oi[:,:3]+=offset
    return dict(i_ext_prev=oi[:,0], i_ext=oi[:,1], i_conf=oi[:,2], dirn=oi[:,3],
                TMV=ov[:,0], T=ov[:,1])

def events_by_group(prices, group, theta):
    p=np.asarray(prices,np.float64); g=np.asarray(group); n=p.shape[0]
    bnd=np.flatnonzero(np.concatenate(([True],g[1:]!=g[:-1]))); bnd=np.concatenate((bnd,[n]))
    parts=[]
    for k in range(len(bnd)-1):
        a,b=int(bnd[k]),int(bnd[k+1])
        if b-a<3: continue
        parts.append(events(p[a:b],theta,offset=a))
    return {k:np.concatenate([q[k] for q in parts]) for k in parts[0]}


## 2 Data


In [6]:
z   = np.load(PANEL)
P   = z['last'].astype(np.float64)
DAY = z['day'].astype(np.int64)
HMS = (z['time_int'] % 1000000).astype(np.int64)
n   = len(P)
assert np.isfinite(P).all(), 'non-finite price'
assert (np.diff(HMS[DAY==DAY[0]])>0).all(), 'time not increasing within session'

ud, dayid = np.unique(DAY, return_inverse=True); ND = len(ud)
nd  = np.flatnonzero(np.concatenate(([True], DAY[1:]!=DAY[:-1])))
bnd = np.concatenate((nd,[n]))
hi  = np.full(ND,-1e18); lo = np.full(ND,1e18)
np.maximum.at(hi,dayid,P); np.minimum.at(lo,dayid,P); RS = hi-lo

in_break = ((HMS>BREAK[0]) & (HMS<BREAK[1])).sum()
print(f'ticks        : {n:,}')
print(f'sessions     : {ND:,}   {ud.min()} to {ud.max()}')
print(f'price range  : {P.min()} to {P.max()}')
print(f'obs in break : {in_break}   (the midday gap Section 4 relies on)')
print(f'session range: median {np.median(RS):.2f}  mean {RS.mean():.2f} points')


ticks        : 18,089,838
sessions     : 1,340   20210225 to 20260826
price range  : 860.8 to 2115.7
obs in break : 0   (the midday gap Section 4 relies on)
session range: median 16.60  mean 20.68 points


## 3 Legs

`group` is the session unit: directional-change state resets at each group boundary and the range
denominator is computed within the same group. Passing a sub-session id instead of the calendar day
is all that is needed to run the sub-session variants.


In [8]:
def legs(P, GRP, thetas=THETAS):
    P=np.ascontiguousarray(np.asarray(P,np.float64)); GRP=np.asarray(GRP); n=len(P)
    ug,gid=np.unique(GRP,return_inverse=True)
    hi=np.full(len(ug),-1e18); lo=np.full(len(ug),1e18)
    np.maximum.at(hi,gid,P); np.minimum.at(lo,gid,P); RNG=hi-lo
    out=[]
    for TH in thetas:
        E=events_by_group(P,GRP,TH)
        ie,ip,ic,dr=E['i_ext'],E['i_ext_prev'],E['i_conf'],E['dirn']
        ok=(ie>ip)&(ic<n-1)&(GRP[ip]==GRP[ic])
        ie,ip,ic,dr=[x[ok] for x in (ie,ip,ic,dr)]
        if len(ie)<2: continue
        dcc=np.concatenate(([-1],ic[:-1])); v=(dcc>ip)&(dcc<ie); v[0]=False
        ie,ip,ic,dr,dcc=[x[v] for x in (ie,ip,ic,dr,dcc)]
        if not len(ie): continue
        out.append(pd.DataFrame(dict(theta=TH, grp=GRP[ie], dirn=dr,
            i_dcc=dcc, i_ext=ie, i_conf=ic,
            delta=TH*np.abs(P[ie]), overshoot=np.abs(P[ie]-P[dcc]),
            retrace=np.abs(P[ic]-P[ie]), session_range=RNG[gid[ie]],
            dur=(ie-ip).astype(np.int64))))
    L=pd.concat(out,ignore_index=True)
    L['ratio']=L.delta/L.session_range
    L['wd']=L.overshoot/L.delta
    return L

L = legs(P, DAY)
print(f'{len(L):,} legs across {len(THETAS)} thresholds')
summ = L.groupby('theta').agg(n=('wd','size'), median_dur_s=('dur','median'), wd=('wd','mean'))
print(summ.to_string(float_format=lambda v: f'{v:.3f}'))


263,372 legs across 6 thresholds
             n  median_dur_s    wd
theta                             
0.0005  172552        55.000 1.590
0.0010   65125       127.000 1.355
0.0020   20573       340.000 1.198
0.0050    3150      1183.500 1.001
0.0070    1439      1776.000 0.907
0.0100     533      2650.000 0.798


## 4 The collapse: cells, two-way ANOVA, compression slope


In [10]:
def cells(L, thetas=THETAS):
    r=[]
    for a,b in zip(EDGES,EDGES[1:]):
        for th in thetas:
            m=(L.ratio>=a)&(L.ratio<b)&(L.theta==th)
            if m.sum()>=MIN_LEGS:
                r.append(dict(band=f'{a:.2f}-{b:.2f}', mid=(a+b)/2, theta=th,
                              mean=float(L.wd[m].mean()), n=int(m.sum())))
    return pd.DataFrame(r)

def collapse(C):
    y=C['mean'].to_numpy(float); one=np.ones((len(C),1))
    X=pd.get_dummies(C[['band','theta']].astype(str),drop_first=True).astype(float)
    def rss(M):
        b,*_=np.linalg.lstsq(M,y,rcond=None); e=y-M@b; return float(e@e),M.shape[1]
    Xb=np.column_stack([one]+[X[k].to_numpy() for k in X if k.startswith('band')])
    Xt=np.column_stack([one]+[X[k].to_numpy() for k in X if k.startswith('theta')])
    Xf=np.column_stack([one]+[X[k].to_numpy() for k in X])
    rb,kb=rss(Xb); rt,kt=rss(Xt); rf,kf=rss(Xf); df2=len(C)-kf
    Fb=((rt-rf)/(kf-kt))/(rf/df2); Ft=((rb-rf)/(kf-kb))/(rf/df2)
    bm=C.groupby('band').agg(mid=('mid','first'),m=('mean','mean')).sort_values('mid')
    w=C.groupby('band')['mean'].agg(['min','max','count']); w['s']=w['max']-w['min']
    sl,ic_,rv,pv,se=stats.linregress(bm['mid'],bm['m'])
    return dict(ncell=len(C), F_band=Fb, p_band=float(1-stats.f.cdf(Fb,kf-kt,df2)),
                F_theta=Ft, p_theta=float(1-stats.f.cdf(Ft,kf-kb,df2)), df=(kf-kt,df2),
                ss_ratio=(rb-rf)/(rt-rf),
                within_between=float(w[w['count']>1]['s'].mean()/(bm.m.max()-bm.m.min())),
                beta_C=float(sl), se=float(se), r2=float(rv**2),
                band_means=bm, monotone=bool((np.diff(bm.m)<0).all()))

C = cells(L); R = collapse(C)
print(C.pivot(index='band',columns='theta',values='mean').round(3).to_string())
print()
for b,r in R['band_means'].iterrows(): print(f'  {b}   omega/delta = {r.m:.3f}')
print(f"\n  band|theta  F{R['df']} = {R['F_band']:.2f}  p = {R['p_band']:.3g}")
print(f"  theta|band  F{R['df']} = {R['F_theta']:.2f}  p = {R['p_theta']:.3f}")
print(f"  SS_theta/SS_band = {R['ss_ratio']:.3f}   within/between = {R['within_between']:.3f}")
print(f"  beta_C = {R['beta_C']:.3f}  (SE {R['se']:.3f}, R2 {R['r2']:.3f})  monotone: {R['monotone']}")


theta      0.0005  0.0010  0.0020  0.0050  0.0070  0.0100
band                                                     
0.05-0.10   1.277   1.260   1.273   1.501   1.399     NaN
0.10-0.15   1.109   1.152   1.176   1.169   1.607     NaN
0.15-0.22   0.970   1.027   1.063   1.132   1.046     NaN
0.22-0.32     NaN   0.915   0.887   0.954   0.994   0.894
0.32-0.45     NaN     NaN   0.724   0.716   0.746   0.773
0.45-0.65     NaN     NaN   0.504   0.517   0.465   0.483

  0.05-0.10   omega/delta = 1.342
  0.10-0.15   omega/delta = 1.243
  0.15-0.22   omega/delta = 1.048
  0.22-0.32   omega/delta = 0.929
  0.32-0.45   omega/delta = 0.740
  0.45-0.65   omega/delta = 0.492

  band|theta  F(5, 17) = 45.13  p = 3.19e-09
  theta|band  F(5, 17) = 2.08  p = 0.118
  SS_theta/SS_band = 0.046   within/between = 0.219
  beta_C = -1.768  (SE 0.112, R2 0.984)  monotone: True


## 5 Figures

Every point is computed in the cell that draws it. There are no pasted constants: that is the specific
defect this notebook exists to prevent.


In [12]:
W,H=520,300
def svg(b,w=W,h=H): return (f'<svg viewBox="0 0 {w} {h}" xmlns="http://www.w3.org/2000/svg" '
    f'font-family="Tinos,Times New Roman,serif" font-size="11">{b}</svg>')
def axes(x0,x1,y0,y1,L=54,Rm=14,T=14,B=34,xt=None,yt=None,xl='',yl='',w=W,h=H):
    X=lambda v:L+(w-L-Rm)*(v-x0)/(x1-x0); Y=lambda v:T+(h-T-B)*(y1-v)/(y1-y0)
    s=[f'<rect x="0" y="0" width="{w}" height="{h}" fill="#fff"/>']
    for v in (yt or []):
        s.append(f'<line x1="{L}" y1="{Y(v):.1f}" x2="{w-Rm}" y2="{Y(v):.1f}" stroke="#e2e2e2" stroke-width=".7"/>')
        s.append(f'<text x="{L-6}" y="{Y(v)+3.5:.1f}" text-anchor="end" fill="#333">{v:g}</text>')
    for v in (xt or []):
        s.append(f'<text x="{X(v):.1f}" y="{h-B+15:.1f}" text-anchor="middle" fill="#333">{v:g}</text>')
    s.append(f'<line x1="{L}" y1="{T}" x2="{L}" y2="{h-B}" stroke="#000" stroke-width="1"/>')
    s.append(f'<line x1="{L}" y1="{h-B}" x2="{w-Rm}" y2="{h-B}" stroke="#000" stroke-width="1"/>')
    if xl: s.append(f'<text x="{(L+w-Rm)/2:.0f}" y="{h-4}" text-anchor="middle" fill="#000">{xl}</text>')
    if yl: s.append(f'<text transform="translate(13,{(T+h-B)/2:.0f}) rotate(-90)" text-anchor="middle" fill="#000">{yl}</text>')
    return s,X,Y
from IPython.display import SVG, display


### Figure 1 — mean total movement against threshold

⟨TMV⟩ = 1 + ⟨ω⟩/δ on the same leg set used everywhere else, so the figure and the tables cannot
disagree. ⟨TMV⟩ = 2 is the GDO scaling law and the zero-profit line for the canonical round trip.


In [14]:
TT=[]
for TH in THETA_FIG:
    E=events_by_group(P,DAY,TH); ie,ip,ic=E['i_ext'],E['i_ext_prev'],E['i_conf']
    ok=(ie>ip)&(ic<n-1)&(DAY[ip]==DAY[ic]); ie,ip,ic=[x[ok] for x in (ie,ip,ic)]
    dcc=np.concatenate(([-1],ic[:-1])); v=(dcc>ip)&(dcc<ie); v[0]=False
    ie,ic,dcc=[x[v] for x in (ie,ic,dcc)]
    TT.append((TH, 1+float((np.abs(P[ie]-P[dcc])/(TH*np.abs(P[ie]))).mean())))
a=[x for x in TT if x[1]>2][-1]; b_=[x for x in TT if x[1]<2][0]
f=(a[1]-2)/(a[1]-b_[1])
CROSS=10**(np.log10(a[0])+f*(np.log10(b_[0])-np.log10(a[0])))
for t,v in TT: print(f'  theta={t:.0e}   <TMV> = {v:.3f}')
print(f'\n  crossing <TMV> = 2 at theta = {CROSS*100:.2f}%')

ymax=max(v for _,v in TT)*1.05
s,X,Y=axes(-3.8,-1.9,1.6,ymax,yt=[2.0,2.5,3.0],xt=[-3.7,-3.3,-3.0,-2.7,-2.3,-2.0],xl='log\u2081\u2080 \u03b8',yl='\u27e8TMV\u27e9')
s.append(f'<line x1="{X(-3.8):.1f}" y1="{Y(2):.1f}" x2="{X(-1.9):.1f}" y2="{Y(2):.1f}" stroke="#000" stroke-dasharray="5 3" stroke-width="1"/>')
s.append(f'<text x="{X(-2.55):.1f}" y="{Y(2)-7:.1f}" font-style="italic">&#10216;TMV&#10217; = 2 (GDO scaling law)</text>')
pts=[(np.log10(t),v) for t,v in TT]
s.append('<polyline points="'+' '.join(f'{X(x):.1f},{Y(y):.1f}' for x,y in pts)+'" fill="none" stroke="#000" stroke-width="1.4"/>')
for x,y in pts: s.append(f'<circle cx="{X(x):.1f}" cy="{Y(y):.1f}" r="3.6" fill="#000"/>')
s.append(f'<text x="{X(-2.35):.1f}" y="{Y(2.35):.1f}" text-anchor="middle" font-size="9.5">crossing at theta &#8776; {CROSS*100:.2f}%</text>')
FIG1=svg(''.join(s)); open('p_fig1.svg','w').write(FIG1); display(SVG(FIG1))


  theta=2e-04   <TMV> = 3.095
  theta=5e-04   <TMV> = 2.590
  theta=1e-03   <TMV> = 2.355
  theta=2e-03   <TMV> = 2.198
  theta=5e-03   <TMV> = 2.001
  theta=7e-03   <TMV> = 1.907
  theta=1e-02   <TMV> = 1.798

  crossing <TMV> = 2 at theta = 0.50%
<IPython.core.display.SVG object>


### Figure 2 — noise-corrected local Hurst exponent

Raw variance ratios are depressed by microstructure noise in the one-second base interval.
Var(r_q) = A·q^b + c is fitted on q ≥ 20 to separate the constant noise floor, which is then removed
before local slopes are taken.


In [16]:
QS=np.array([1,2,3,5,8,13,20,32,50,80,125,200,329,520,820,1141,1800,2616,3600])
V=np.array([float(np.var((P[q:]-P[:-q])[DAY[q:]==DAY[:-q]])) for q in QS])
m=QS>=20; ff=lambda q,A,bb,c: A*q**bb+c
(A_,B_,Cn),_=curve_fit(ff,QS[m].astype(float),V[m],p0=[V[m][0]/QS[m][0],1.,0.],maxfev=40000)
lq=np.log(QS); lv=np.log(np.maximum(V-Cn,1e-12))
HQ=[(int(QS[i]),(lv[i+1]-lv[i-1])/(lq[i+1]-lq[i-1])/2) for i in range(1,len(QS)-1) if QS[i]>=5]
HMEAN=float(np.mean([h for _,h in HQ])); HMIN=min(h for _,h in HQ); HMAX=max(h for _,h in HQ)
print(f'  b = {B_:.4f}  ->  H = {B_/2:.4f}')
print(f'  noise floor c = {Cn:.4f} pt^2 = {np.sqrt(max(Cn,0)/2):.4f} points per side')
print(f'  local H: {HMIN:.3f} to {HMAX:.3f}, mean {HMEAN:.3f}')
def H_at(q): return min(HQ,key=lambda x:abs(np.log(x[0])-np.log(q)))[1]
d_short=summ.median_dur_s.iloc[0]; d_long=summ.median_dur_s.iloc[-1]
print(f'  at {d_short:.0f}s (wd={summ.wd.iloc[0]:.2f}): H={H_at(d_short):.3f}   '
      f'at {d_long:.0f}s (wd={summ.wd.iloc[-1]:.2f}): H={H_at(d_long):.3f}')

s,X,Y=axes(0.6,3.6,0.40,0.56,yt=[0.42,0.46,0.50,0.54],xt=[1,2,3],xl='log\u2081\u2080 horizon q (seconds)',yl='local Hurst H')
s.append(f'<line x1="{X(0.6):.1f}" y1="{Y(0.5):.1f}" x2="{X(3.6):.1f}" y2="{Y(0.5):.1f}" stroke="#000" stroke-dasharray="5 3" stroke-width="1"/>')
s.append(f'<text x="{X(2.9):.1f}" y="{Y(0.5)-6:.1f}" font-style="italic">H = 0.5 (martingale)</text>')
pts=[(np.log10(q),h) for q,h in HQ]
s.append('<polyline points="'+' '.join(f'{X(x):.1f},{Y(y):.1f}' for x,y in pts)+'" fill="none" stroke="#000" stroke-width="1.3"/>')
for x,y in pts: s.append(f'<circle cx="{X(x):.1f}" cy="{Y(y):.1f}" r="2.6" fill="#000"/>')
s.append(f'<text x="{X(2.1):.1f}" y="{Y(0.415):.1f}" text-anchor="middle" font-size="9.5">no monotone trend: H fluctuates about {HMEAN:.2f}</text>')
FIG2=svg(''.join(s)); open('p_fig2.svg','w').write(FIG2); display(SVG(FIG2))


  b = 0.9500  ->  H = 0.4750
  noise floor c = 0.0123 pt^2 = 0.0783 points per side
  local H: 0.432 to 0.526, mean 0.484
  at 55s (wd=1.59): H=0.489   at 2650s (wd=0.80): H=0.483
<IPython.core.display.SVG object>


### Figure 3 — the collapse


In [18]:
COL=['#444','#666','#111','#333','#555','#222']
MK=['circle','square','triangle','diamond','plus','cross']
def marker(s,mk,col,x,y):
    if mk=='circle': s.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3.4" fill="#fff" stroke="{col}" stroke-width="1.3"/>')
    elif mk=='square': s.append(f'<rect x="{x-3:.1f}" y="{y-3:.1f}" width="6" height="6" fill="#fff" stroke="{col}" stroke-width="1.3"/>')
    elif mk=='triangle': s.append(f'<polygon points="{x:.1f},{y-3.8:.1f} {x-3.4:.1f},{y+2.6:.1f} {x+3.4:.1f},{y+2.6:.1f}" fill="#fff" stroke="{col}" stroke-width="1.3"/>')
    elif mk=='diamond': s.append(f'<polygon points="{x:.1f},{y-4:.1f} {x+3.4:.1f},{y:.1f} {x:.1f},{y+4:.1f} {x-3.4:.1f},{y:.1f}" fill="#fff" stroke="{col}" stroke-width="1.3"/>')
    elif mk=='plus': s.append(f'<path d="M{x-3.6:.1f},{y:.1f}h7.2M{x:.1f},{y-3.6:.1f}v7.2" stroke="{col}" stroke-width="1.5"/>')
    else: s.append(f'<path d="M{x-3:.1f},{y-3:.1f}l6,6M{x+3:.1f},{y-3:.1f}l-6,6" stroke="{col}" stroke-width="1.5"/>')

s,X,Y=axes(0.02,0.62,0.3,1.75,yt=[0.5,0.75,1.0,1.25,1.5,1.75],xt=[0.1,0.2,0.3,0.4,0.5,0.6],
           xl='\u03b4 / session range',yl='\u27e8\u03c9\u27e9 / \u03b4')
s.append(f'<line x1="{X(0.02):.1f}" y1="{Y(1):.1f}" x2="{X(0.62):.1f}" y2="{Y(1):.1f}" stroke="#000" stroke-width="1" stroke-dasharray="5 3"/>')
s.append(f'<text x="{X(0.42):.1f}" y="{Y(1)-6:.1f}" fill="#000" font-style="italic">GDO scaling law: &#10216;&#969;&#10217; = &#948;</text>')
for j,th in enumerate(THETAS):
    g=C[C.theta==th].sort_values('mid')
    if not len(g): continue
    pts=[(r.mid,r['mean']) for _,r in g.iterrows()]
    s.append('<polyline points="'+' '.join(f'{X(x):.1f},{Y(y):.1f}' for x,y in pts)+f'" fill="none" stroke="{COL[j]}" stroke-width="1" opacity=".55"/>')
    for x0,y0 in pts: marker(s,MK[j],COL[j],X(x0),Y(y0))
s.append('<rect x="380" y="18" width="126" height="86" fill="#fff" stroke="#bbb" stroke-width=".7"/>')
for j,th in enumerate(THETAS):
    yy=30+13*j; s.append(f'<text x="402" y="{yy}" fill="#000" font-size="9.5">theta = {th:.0e}</text>')
    marker(s,MK[j],COL[j],392.0,yy-3.5)
FIG3=svg(''.join(s)); open('p_fig3.svg','w').write(FIG3); display(SVG(FIG3))


<IPython.core.display.SVG object>


## 6 Is the collapse an artefact of its own denominator?

The session range contains the leg being measured, so a large overshoot can widen R, lower δ/R, and
induce a negative association mechanically. Two pre-registered tests, both frozen and timestamped
before being run: `PREREG_detached_denominator_v1.0.md` and `PREREG_leave_one_leg_out_v1.0.md`.


In [20]:
inA=(HMS>=SESSION[0])&(HMS<=BREAK[0]); inB=(HMS>=BREAK[1])&(HMS<=SESSION[1])
def rng_of(mask):
    h=np.full(ND,-1e18); l=np.full(ND,1e18)
    np.maximum.at(h,dayid[mask],P[mask]); np.minimum.at(l,dayid[mask],P[mask]); return h-l
RA,RB=rng_of(inA),rng_of(inB); good=(RA>0)&(RB>0)&np.isfinite(RA)&np.isfinite(RB)
print(f'median R_morning {np.median(RA[good]):.2f}   R_afternoon {np.median(RB[good]):.2f}')

det=[]
for TH in THETAS:
    g=L[L.theta==TH]
    wA=inA[g.i_dcc.values]&inA[g.i_conf.values]; wB=inB[g.i_dcc.values]&inB[g.i_conf.values]
    keep=(wA|wB)&good[dayid[g.i_ext.values]]
    gg=g[keep]; wA=wA[keep]; gi=dayid[gg.i_ext.values]
    own=np.where(wA,RA[gi],RB[gi]); oth=np.where(wA,RB[gi],RA[gi])
    det.append(pd.DataFrame(dict(theta=TH,win=np.where(wA,'A','B'),wd=gg.wd.values,
                                 x_own=gg.delta.values/own, x_det=gg.delta.values/oth)))
D=pd.concat(det,ignore_index=True)
print(f'{len(D):,} legs lie entirely inside one window')
for th in THETAS:
    mm=D.theta==th
    print(f'  theta={th:.0e}  rho attached {stats.spearmanr(D.x_own[mm],D.wd[mm]).statistic:+.3f}'
          f'   detached {stats.spearmanr(D.x_det[mm],D.wd[mm]).statistic:+.3f}')
for w in ('A','B'):
    S=D[D.win==w]
    ro=stats.spearmanr(S.x_own,S.wd).statistic; rd=stats.spearmanr(S.x_det,S.wd).statistic
    print(f'  window {w}: retained {rd/ro*100:.0f}%')
def curve_of(df,col):
    T=df.rename(columns={col:'ratio'})[['theta','ratio','wd']]
    return collapse(cells(T))
R_own=curve_of(D,'x_own'); R_det=curve_of(D,'x_det')
DET_BETA=R_det['beta_C']; print(f"  beta_C attached {R_own['beta_C']:.3f}  detached {DET_BETA:.3f}")


median R_morning 10.20   R_afternoon 12.60
258,489 legs lie entirely inside one window
  theta=5e-04  rho attached -0.181   detached -0.130
  theta=1e-03  rho attached -0.139   detached -0.086
  theta=2e-03  rho attached -0.149   detached -0.097
  theta=5e-03  rho attached -0.271   detached -0.151
  theta=7e-03  rho attached -0.350   detached -0.194
  theta=1e-02  rho attached -0.354   detached -0.152
  window A: retained 84%
  window B: retained 87%
  beta_C attached -1.836  detached -1.028


In [21]:
# leave-one-leg-out: session range recomputed with the focal leg's own ticks removed
pmax=np.empty(n); pmin=np.empty(n); smax=np.empty(n); smin=np.empty(n)
ds=np.zeros(n,np.int64); de=np.zeros(n,np.int64)
for a,b in zip(bnd[:-1],bnd[1:]):
    pmax[a:b]=np.maximum.accumulate(P[a:b]); pmin[a:b]=np.minimum.accumulate(P[a:b])
    smax[a:b]=np.maximum.accumulate(P[a:b][::-1])[::-1]; smin[a:b]=np.minimum.accumulate(P[a:b][::-1])[::-1]
    ds[a:b]=a; de[a:b]=b-1
NEG,POS=-1e18,1e18
Lo=L.reset_index(drop=True).copy(); ie=Lo.i_ext.values; l0=Lo.i_dcc.values; r0=Lo.i_conf.values
lft=l0>ds[ie]; rgt=r0<de[ie]
mx=np.where(lft,pmax[np.maximum(l0-1,ds[ie])],NEG); mn=np.where(lft,pmin[np.maximum(l0-1,ds[ie])],POS)
mx=np.maximum(mx,np.where(rgt,smax[np.minimum(r0+1,de[ie])],NEG))
mn=np.minimum(mn,np.where(rgt,smin[np.minimum(r0+1,de[ie])],POS))
Lo['R_loo']=np.where(lft|rgt,mx-mn,0.0)
Lo=Lo[Lo.R_loo>0].reset_index(drop=True).copy(); Lo['x_loo']=Lo.delta/Lo.R_loo
ch=Lo.R_loo<Lo.session_range
print(f'legs whose denominator changes: {int(ch.sum()):,} / {len(Lo):,} = {ch.mean()*100:.1f}%')
print(f'  their <omega>/delta {Lo.wd[ch].mean():.3f}  vs unaffected {Lo.wd[~ch].mean():.3f}')
R_loo=collapse(cells(Lo[['theta','x_loo','wd']].rename(columns={'x_loo':'ratio'})))
LOO_BETA=R_loo['beta_C']; LOO_PT=R_loo['p_theta']
print()
print(f"{'denominator':>26} | {'beta_C':>7} | {'p_theta':>8} | {'SS_t/SS_b':>9}")
for nm,rr in (('same-session (published)',R),('opposite subsession',R_det),('leave-one-leg-out',R_loo)):
    print(f"{nm:>26} | {rr['beta_C']:>7.3f} | {rr['p_theta']:>8.3f} | {rr['ss_ratio']:>9.3f}")
print(f'\n  flattening under LOO: {(abs(R["beta_C"])-abs(LOO_BETA))/abs(R["beta_C"])*100:.1f}%')


legs whose denominator changes: 7,135 / 263,372 = 2.7%
  their <omega>/delta 2.209  vs unaffected 1.469

               denominator |  beta_C |  p_theta | SS_t/SS_b
  same-session (published) |  -1.768 |    0.118 |     0.046
       opposite subsession |  -1.028 |    0.000 |     0.889
         leave-one-leg-out |  -1.454 |    0.555 |     0.036

  flattening under LOO: 17.8%


## 7 The strategy

The reversal rule fades the direction just confirmed: at each confirmation take the opposite position
and hold it to the exit rule. Per round trip the realised move is retrace − ω = (retrace − δ) + (δ − ω),
which is why the scaling law is this rule's null hypothesis rather than its background theory.

Twenty-four cells: four thresholds × two entry windows × three exit rules. The rule reads no profit and
loss anywhere in its construction.


In [23]:
FADE_TH   = (3e-3, 5e-3, 7e-3, 1e-2, 1.2e-2)   # 5 x 2 x 3 = 30 cells, as designed
FADE_WIN  = {'full': 0, '09:45+': 94500}
FADE_EXIT = {'to conf': None, '<=60m': 3600, '<=120m': 7200}

PCHG=np.zeros(n)
for a,b in zip(bnd[:-1],bnd[1:]): PCHG[a:b-1]=P[a+1:b]-P[a:b-1]

def fade_position(theta, win_hms, max_hold):
    """Explicit position array, flat between trades. Entry one second after the confirmation."""
    E=events_by_group(P,DAY,theta); ie,ip,ic,dr=E['i_ext'],E['i_ext_prev'],E['i_conf'],E['dirn']
    ok=(ie>ip)&(ic<n-1)&(DAY[ip]==DAY[ic]); ie,ip,ic,dr=[x[ok] for x in (ie,ip,ic,dr)]
    dcc=np.concatenate(([-1],ic[:-1])); v=(dcc>ip)&(dcc<ie); v[0]=False
    ie,ic,dr,dcc=[x[v] for x in (ie,ic,dr,dcc)]
    e0=dcc+DELAY
    keep=(e0<n)&(HMS[np.minimum(e0,n-1)]>win_hms)&(DAY[np.minimum(e0,n-1)]==DAY[ie])
    e0,ic,dr,ie=[x[keep] for x in (e0,ic,dr,ie)]
    e1=ic+DELAY
    if max_hold is not None: e1=np.minimum(e1, e0+max_hold)
    e1=np.minimum(e1, de[np.minimum(ie,n-1)])          # never past the session close
    pos=np.zeros(n)
    for s0,s1,d in zip(e0,e1,dr):
        if s1>s0: pos[s0:s1]=-d
    return pos, int((e1>e0).sum())

def score(pos, trips, cost=COST, sel=None):
    t=np.abs(np.diff(pos,prepend=0.0)); gr=pos*PCHG
    daily=np.bincount(dayid,gr-t*cost,ND); dg=np.bincount(dayid,gr,ND)
    dt=np.bincount(dayid,t,ND)
    if sel is None: sel=np.ones(ND,bool)
    s=daily[sel]; eq=np.cumsum(s); T=float(dt[sel].sum()); G=float(dg[sel].sum())
    mdd=float(np.max(np.maximum.accumulate(eq)-eq)) if len(eq) else np.nan
    return dict(net=float(s.sum()), gross=G, turn=T, trips=trips,
                be=G/max(T,1e-9), mdd=mdd,
                sharpe=float(s.mean()/s.std()*np.sqrt(252)) if s.std()>0 else np.nan,
                calmar=float(s.sum()/mdd) if mdd and mdd>0 else np.nan, daily=daily)

GRID={}
for th in FADE_TH:
    for wn,wv in FADE_WIN.items():
        for en,ev in FADE_EXIT.items():
            # %g, not %.0e: under %.0e the labels for 1e-2 and 1.2e-2 collide
            GRID[f'{th:g}|{wn}|{en}']=fade_position(th,wv,ev)
print(f'{len(GRID)} cells built')
tab=pd.DataFrame({k:{kk:vv for kk,vv in score(*v).items() if kk!='daily'} for k,v in GRID.items()}).T
print(tab.sort_values('net',ascending=False).head(8).to_string(float_format=lambda x:f'{x:.2f}'))


30 cells built
                         net   gross    turn   trips   be    mdd  sharpe  calmar
0.007|09:45+|to conf 1855.15 2316.80 2638.00 1319.00 0.88 154.40    2.14   12.02
0.007|full|to conf   1796.55 2300.20 2878.00 1439.00 0.80 201.70    1.95    8.91
0.007|09:45+|<=120m  1756.55 2218.20 2638.00 1319.00 0.84 154.40    2.04   11.38
0.01|full|to conf    1743.85 1930.40 1066.00  533.00 1.81 104.05    2.51   16.76
0.007|full|<=120m    1660.95 2164.60 2878.00 1439.00 0.75 201.70    1.81    8.23
0.01|09:45+|to conf  1660.75 1832.60  982.00  491.00 1.87 104.05    2.43   15.96
0.01|full|<=120m     1636.35 1822.90 1066.00  533.00 1.71  99.75    2.36   16.40
0.01|09:45+|<=120m   1587.45 1759.30  982.00  491.00 1.79  99.75    2.33   15.91


### Cost sensitivity

Break-even cost is derived, not assumed: it is the cost per leg at which net profit reaches zero.


In [25]:
COSTS=[0,0.0875,0.175,0.2965,0.4,0.6,1.0]
best=tab.net.idxmax()
row={f'{c:g}':score(*GRID[best],cost=c)['net'] for c in COSTS}
print(f'best cell: {best}')
print(pd.Series(row).to_string(float_format=lambda x:f'{x:+.0f}'))
print(f"\nbreak-even {tab.loc[best,'be']:.3f} points per leg")


best cell: 0.007|09:45+|to conf
0        +2317
0.0875   +2086
0.175    +1855
0.2965   +1535
0.4      +1262
0.6       +734
1         -321

break-even 0.878 points per leg


## 8 Anchored walk-forward

Each fold selects its cell from every session preceding the test period and is scored on that period
alone, so selection is inside the test rather than before it. WFE is Pardo's walk-forward efficiency:
out-of-sample net per session divided by in-sample net per session.


In [27]:
FOLDS=[('2023',20230101,20240101),('2024',20240101,20250101),('2025',20250101,20260101),
       ('2026H1',20260101,20260425),('2026H2',20260425,20270101)]
rows=[]; stitched=np.zeros(ND); cov=np.zeros(ND,bool)
for name,a,b in FOLDS:
    tr=ud<a; te=(ud>=a)&(ud<b)
    if tr.sum()<50 or te.sum()<5: continue
    sc={k:score(*v,sel=tr) for k,v in GRID.items()}
    pick=max(sc,key=lambda k: (sc[k]['calmar'] if np.isfinite(sc[k]['calmar']) else -9e9))
    IS=sc[pick]; OS=score(*GRID[pick],sel=te)
    wfe=(OS['net']/max(te.sum(),1))/((IS['net']/max(tr.sum(),1)) or np.nan)
    stitched[te]=OS['daily'][te]; cov|=te
    rows.append(dict(fold=name,train=int(tr.sum()),cell=pick,IS_sharpe=IS['sharpe'],
                     OOS_sharpe=OS['sharpe'],OOS_net=OS['net'],MDD=OS['mdd'],WFE=wfe,trips=OS['trips']))
WF=pd.DataFrame(rows)
print(WF.to_string(index=False,float_format=lambda x:f'{x:.2f}'))
st=stitched[cov]; eq=np.cumsum(st)
WF_SHARPE=float(st.mean()/st.std()*np.sqrt(252))
WF_NET=float(st.sum()); WF_MDD=float(np.max(np.maximum.accumulate(eq)-eq))
print(f'\nstitched out-of-sample: net {WF_NET:+.0f}  Sharpe {WF_SHARPE:.2f}  maxDD {WF_MDD:.0f}'
      f'  positive folds {int((WF.OOS_net>0).sum())}/{len(WF)}')


  fold  train               cell  IS_sharpe  OOS_sharpe  OOS_net   MDD  WFE  trips
  2023    436 0.01|09:45+|<=120m       3.91        3.99   280.75 29.00 0.62    491
  2024    681 0.01|09:45+|<=120m       3.83        1.94    61.30 13.15 0.15    491
  2025    931 0.01|09:45+|<=120m       3.39        1.32   297.30 99.75 0.97    491
2026H1   1180  0.012|full|<=120m       2.09        3.97   162.90 10.00 2.12    304
2026H2   1255  0.012|full|<=120m       2.21        3.03    60.65  0.00 0.65    304

stitched out-of-sample: net +863  Sharpe 1.79  maxDD 100  positive folds 5/5


## 9 Section 7 — is the state variable actionable?

E1 asks whether the session range is causally predictable; E2 whether the regime it defines can be
classified in time to act on it; E3 whether acting on it beats a constant threshold. E3 was frozen in
`PREREG_E3_threshold_control_v1.0.md` before it was computed and run once.


In [29]:
dd=pd.DataFrame(dict(p=P,day=DAY,hms=HMS)); gp=dd.groupby('day')['p']
DF=pd.DataFrame(dict(R=gp.max()-gp.min())).reset_index()
DF['med20']=DF.R.shift(1).rolling(20).median()
BURN=120
def rhat(cut):
    q=dd[dd.hms<=cut].groupby('day')['p']
    d2=DF.merge((q.max()-q.min()).clip(lower=0.05).rename('OR').reset_index(),on='day')
    d2=d2.merge(q.last().rename('Pdec').reset_index(),on='day').dropna().reset_index(drop=True)
    x,z2,y=np.log(d2.OR),np.log(d2.med20),np.log(d2.R); out=np.full(len(d2),np.nan)
    fin=np.isfinite(x)&np.isfinite(z2)&np.isfinite(y)
    for t in range(BURN,len(d2)):
        k=fin[:t]; M=np.column_stack([np.ones(k.sum()),x[:t][k],z2[:t][k]])
        bb,*_=np.linalg.lstsq(M,y[:t][k],rcond=None)
        out[t]=np.exp(bb[0]+bb[1]*x.iloc[t]+bb[2]*z2.iloc[t])
    d2['Rhat']=out; return d2.dropna().reset_index(drop=True)

# both cutoffs are scored on the SAME sessions, so the comparison is like for like
RH={c:rhat(c) for c in (94500,103000)}
common=set(RH[94500].day) & set(RH[103000].day)
print(f'{len(common):,} sessions carry a causal estimate at both cutoffs')
for cut in (94500,103000):
    d2=RH[cut]; d2=d2[d2.day.isin(common)]
    rho=stats.spearmanr(d2.Rhat,d2.R).statistic
    r2=1-((np.log(d2.R)-np.log(d2.Rhat))**2).sum()/((np.log(d2.R)-np.log(d2.R).mean())**2).sum()
    print(f'E1 cutoff {cut//10000:02d}:{cut//100%100:02d}  Spearman {rho:.3f}  R2(log) {r2:.3f}')
    if cut==94500: E1_RHO_0945=rho
E1_RHO=rho


1,200 sessions carry a causal estimate at both cutoffs
E1 cutoff 09:45  Spearman 0.695  R2(log) 0.460
E1 cutoff 10:30  Spearman 0.759  R2(log) 0.563


In [30]:
# E2 - does the estimate put the leg on the right side of the crossing?
d2=rhat(103000); Rh=dict(zip(d2.day,d2.Rhat)); use=set(d2.day)
S=L[L.grp.isin(use)].copy()
S['hms_in']=HMS[S.i_dcc.values]
S=S[S.hms_in>103000].copy()
S['rt']=S.delta/S.session_range
S['rh']=S.delta/S.grp.map(Rh).values
def band_curve(x,y):
    return [(f'{a:.2f}-{b:.2f}', float(y[(x>=a)&(x<b)].mean()) if ((x>=a)&(x<b)).sum()>=MIN_LEGS else np.nan)
            for a,b in zip(EDGES,EDGES[1:])]
print(f'{len(S):,} legs entered after 10:30')
print(f"{'band':>12} | {'true':>7} | {'estimated':>10}")
for (b1,v1),(b2,v2) in zip(band_curve(S.rt,S.wd),band_curve(S.rh,S.wd)):
    print(f'{b1:>12} | {v1:>7.3f} | {v2:>10.3f}')
yh=S.rh>=0.20
E2_FADE=float(S.wd[yh].mean())
print(f'\npredicted-fade bucket realises <omega>/delta = {E2_FADE:.3f}   (no conditioning: {S.wd.mean():.3f})')


172,955 legs entered after 10:30
        band |    true |  estimated
   0.05-0.10 |   1.265 |      1.290
   0.10-0.15 |   1.145 |      1.205
   0.15-0.22 |   1.014 |      1.073
   0.22-0.32 |   0.882 |      0.977
   0.32-0.45 |   0.726 |      0.801
   0.45-0.65 |   0.478 |      0.745

predicted-fade bucket realises <omega>/delta = 0.937   (no conditioning: 1.492)


In [31]:
# E3 - causal threshold control, frozen spec, run once
TARGET=0.32
d2=rhat(103000); d2['theta_star']=TARGET*d2.Rhat/d2.Pdec
TH_DAY=dict(zip(d2.day,d2.theta_star)); USE=set(d2.day)
def e3_arm(theta_of_day):
    pos=np.zeros(n); wd=0.0; nl=0
    for a,b in zip(bnd[:-1],bnd[1:]):
        day=int(DAY[a])
        if day not in USE: continue
        th=theta_of_day(day)
        s0=a+int(np.searchsorted(HMS[a:b],103000,'right'))
        if b-s0<10 or not np.isfinite(th) or th<=0: continue
        E=events(P[s0:b],th,offset=s0)
        ie,ip,ic,dr=E['i_ext'],E['i_ext_prev'],E['i_conf'],E['dirn']
        ok=(ie>ip)&(ic<b-1); ie,ip,ic,dr=[x[ok] for x in (ie,ip,ic,dr)]
        if len(ie)<2: continue
        dcc=np.concatenate(([-1],ic[:-1])); v=(dcc>ip)&(dcc<ie); v[0]=False
        ie,ic,dr,dcc=[x[v] for x in (ie,ic,dr,dcc)]
        if not len(ie): continue
        for s1,s2,d in zip(dcc+DELAY,np.minimum(ic+DELAY,b-1),dr):
            if s2>s1: pos[s1:s2]=-d
        wd+=float((np.abs(P[ie]-P[dcc])/(th*np.abs(P[ie]))).sum()); nl+=len(ie)
    return pos,nl,wd/max(nl,1)
sel=np.isin(ud,list(USE))
res=[]
for nm,fn in [('adaptive 0.32*Rhat/P',lambda d:TH_DAY[d])]+[(f'fixed {t:.0e}',(lambda d,t=t:t)) for t in THETAS]:
    pos,nl,w=e3_arm(fn); s_=score(pos,nl,sel=sel)
    res.append(dict(arm=nm,net=s_['net'],sharpe=s_['sharpe'],mdd=s_['mdd'],be=s_['be'],trips=nl,wd=w))
E3=pd.DataFrame(res)
print(E3.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
bestfix=E3[E3.arm!='adaptive 0.32*Rhat/P'].sort_values('net').iloc[-1]
E3_ADAPT=float(E3.iloc[0].net); E3_BEST=float(bestfix.net)
r=stats.spearmanr(E3[E3.arm!='adaptive 0.32*Rhat/P'].wd, E3[E3.arm!='adaptive 0.32*Rhat/P'].be)
E3_RHO=float(r.statistic)
print(f'\nbest fixed: {bestfix.arm}  net {E3_BEST:+.0f}   adaptive net {E3_ADAPT:+.0f}')
print(f'Spearman( E[omega/delta] , break-even ) over the six fixed arms = {E3_RHO:.3f}  p={r.pvalue:.4f}')


                 arm        net  sharpe       mdd     be  trips    wd
adaptive 0.32*Rhat/P   1193.600   1.532   188.950  0.483   1940 0.936
         fixed 5e-04 -71838.950 -15.138 71828.100 -0.143 112863 1.598
         fixed 1e-03 -28181.300 -11.281 28185.250 -0.156  42574 1.359
         fixed 2e-03  -7451.300  -5.552  7455.100 -0.104  13360 1.184
         fixed 5e-03    707.900   0.898   378.500  0.363   1882 0.978
         fixed 7e-03   1583.150   2.346   103.600  1.110    847 0.837
         fixed 1e-02   1290.400   2.272    69.900  2.311    302 0.717

best fixed: fixed 7e-03  net +1583   adaptive net +1194
Spearman( E[omega/delta] , break-even ) over the six fixed arms = -0.943  p=0.0048


## 10 Consistency check against the manuscript

This cell exists because of a specific failure. Before it was written, three figures had been generated
from an older data panel while every table used the current one, and nothing in the workflow could
notice. Each row below pairs a number printed in the paper with the number this notebook just computed.
A mismatch stops the run.

Rows marked `blend` are not checked here: they combine the reversal rule with a fitted classifier whose
artefacts are outside this notebook, so asserting them would be checking a stored value against itself.


In [33]:
CHECKS = [
  ('sessions',                     ND,                    1340,    0),
  ('ticks',                        n,                     18089838,0),
  ('cells at or above 60 legs',    R['ncell'],            28,      0),
  ('band mean 0.05-0.10',          R['band_means'].m.iloc[0], 1.342, 0.001),
  ('band mean 0.45-0.65',          R['band_means'].m.iloc[-1],0.492, 0.001),
  ('F band | theta',               R['F_band'],           45.13,   0.05),
  ('p theta | band',               R['p_theta'],          0.118,   0.002),
  ('SS_theta / SS_band',           R['ss_ratio'],         0.046,   0.002),
  ('within / between',             R['within_between'],   0.219,   0.002),
  ('compression slope beta_C',     R['beta_C'],          -1.768,   0.002),
  ('TMV crossing, per cent',       CROSS*100,             0.50,    0.01),
  ('Hurst b',                      B_,                    0.950,   0.002),
  ('Hurst H',                      B_/2,                  0.475,   0.001),
  ('noise floor, pt^2',            Cn,                    0.0123,  0.0005),
  ('local H mean',                 HMEAN,                 0.48,    0.005),
  ('detached beta_C',              DET_BETA,             -1.028,   0.005),
  ('leave-one-out beta_C',         LOO_BETA,             -1.454,   0.005),
  ('leave-one-out p theta',        LOO_PT,                0.555,   0.005),
  ('legs with changed denominator, %', ch.mean()*100,     2.7,     0.1),
  ('E1 Spearman at 09:45',         E1_RHO_0945,           0.695,   0.005),
  ('E1 Spearman at 10:30',         E1_RHO,                0.759,   0.005),
  ('E2 predicted-fade <w>/d',      E2_FADE,               0.937,   0.005),
  ('E3 Spearman diagnostic-breakeven', E3_RHO,           -0.943,   0.005),
  ('reversal walk-forward net',     WF_NET,                863,     2),
  ('reversal walk-forward Sharpe',  WF_SHARPE,             1.79,    0.02),
]
bad=0
print(f"{'quantity':>36} | {'computed':>12} | {'paper':>10} | verdict")
for name,got,want,tol in CHECKS:
    ok=abs(float(got)-float(want))<=tol
    bad+=not ok
    print(f'{name:>36} | {float(got):>12.4f} | {float(want):>10.4f} | {"ok" if ok else "MISMATCH"}')
print()
print(f'reversal-rule walk-forward: net {WF_NET:+.0f}, Sharpe {WF_SHARPE:.2f} '
      f'(blend figures in Table 3 use the frozen classifier artefacts and are not asserted)')
assert bad==0, f'{bad} figure(s) in the manuscript disagree with this notebook'
print('\nAll checked figures agree with the manuscript.')


                            quantity |     computed |      paper | verdict
                            sessions |    1340.0000 |  1340.0000 | ok
                               ticks | 18089838.0000 | 18089838.0000 | ok
           cells at or above 60 legs |      28.0000 |    28.0000 | ok
                 band mean 0.05-0.10 |       1.3419 |     1.3420 | ok
                 band mean 0.45-0.65 |       0.4923 |     0.4920 | ok
                      F band | theta |      45.1313 |    45.1300 | ok
                      p theta | band |       0.1176 |     0.1180 | ok
                  SS_theta / SS_band |       0.0462 |     0.0460 | ok
                    within / between |       0.2190 |     0.2190 | ok
            compression slope beta_C |      -1.7685 |    -1.7680 | ok
              TMV crossing, per cent |       0.5014 |     0.5000 | ok
                             Hurst b |       0.9500 |     0.9500 | ok
                             Hurst H |       0.4750 |     0.4750 | ok
           

---

### Provenance

| document | SHA-256 |
|---|---|
| `PREREG_external_test_v0.1.md` | `2b7760fe…c397347` |
| `PREREG_external_test_v0.2_amendment.md` | `88d851ae…2b8ebbab` |
| `PREREG_external_test_v0.3_amendment.md` | `e03a2407…d44a1eb89` |
| `PREREG_E3_threshold_control_v1.0.md` | `278341d4…c374c70` |
| `PREREG_detached_denominator_v1.0.md` | `4450429f…700e9b7d` |
| `PREREG_leave_one_leg_out_v1.0.md` | `bac0007b…b96d676` |
| `PREREG_E5_remaining_space_v1.0.md` | `d4d68448…b20699c9b` |

Each was hashed and anchored in the Bitcoin blockchain through OpenTimestamps before the work it
governs was run. `ots verify <file>.ots` checks the claim without trusting the author.
